# HW1-B. A Gated Activation Function

## About this notebook

This notebook is part of HW1 for the 50.039 Deep Learning course at the Singapore University of Technology and Design.

**Author:** Matthieu DE MARI (matthieu_demari@sutd.edu.sg)

**Version:** 1.0 (2026)

**Requirements:**
- Python 3
- Matplotlib
- Numpy
- Pandas
- PyTorch
- Torchmetrics

## 0. Imports and CUDA

In [ ]:
# Matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
# Numpy
import numpy as np
# Pandas
import pandas as pd
# Torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import BinaryAccuracy
# Helper functions (additional file)
from helper_functions import *

In [ ]:
# Use GPU if available, else use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 4. Writing a Gated Activation Layer

In this section, we will implement a **Gated Linear Unit (GLU)** variant. This is a powerful building block used in many modern architectures.

### Mathematical Definition

The Gated Layer computes:

$$f(x) = \text{Linear}_1(x) \odot \sigma(\text{Linear}_2(x))$$

Where:
- $\text{Linear}_1(x) = W_1 x + b_1$ is the **value path** (what information to pass)
- $\text{Linear}_2(x) = W_2 x + b_2$ is the **gate path** (how much to pass)
- $\sigma$ is the **sigmoid function**: $\sigma(z) = \frac{1}{1 + e^{-z}}$
- $\odot$ denotes **element-wise multiplication**

### Intuition

Think of it like a water valve:
- The **value path** determines the water pressure (the information)
- The **gate path** determines how open the valve is (0 = closed, 1 = fully open)
- The output is the actual water flow (information that passes through)

The gate learns to selectively allow or block information based on the input!

---

**Question 7:** Consider the gating mechanism $\sigma(\text{Linear}_2(x))$:

- If the gate path outputs all **0.5** for every element, what happens to the output? How does this compare to simply halving the value path?
- If the gate path outputs all **1s**, what happens?
- If the gate path outputs all **0s**, what happens?
- Why might the network learn to use intermediate gate values (between 0 and 1)? How is this better than simply using a single linear combined with a sigmoid?

---

---

**Question 8:** Regarding differentiability and gradient flow:

- Is the sigmoid function $\sigma(z)$ differentiable everywhere? What is its derivative?
- Is element-wise multiplication a differentiable operation?
- Can PyTorch's autograd compute gradients for all parameters $(W_1, b_1, W_2, b_2)$?
- Write out the gradient $\frac{\partial f}{\partial W_1}$ in terms of the other components (you can use the chain rule).

---

## 5. Implementing the GatedLayer

Now let's implement the `GatedLayer` class. Study the code structure below - there are several `None` values that need to be replaced.

**Hints:**
- Use `nn.Linear(input_dim, output_dim)` for the linear transformations
- Use `nn.Sigmoid()` for the sigmoid activation
- Use `*` for element-wise multiplication in PyTorch

In [ ]:
class GatedLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        
        # Value path: Linear transformation for the "content"
        self.linear_value = None  # TODO: Create nn.Linear for value path
        
        # Gate path: Linear transformation followed by sigmoid for the "gate"
        self.linear_gate = None   # TODO: Create nn.Linear for gate path
        self.sigmoid = None       # TODO: Create nn.Sigmoid()
    
    def forward(self, x):
        # Compute value path
        value = None  # TODO: Pass x through linear_value
        
        # Compute gate path (linear + sigmoid)
        gate = None   # TODO: Pass x through linear_gate, then sigmoid
        
        # Element-wise multiplication: value * gate
        output = None  # TODO: Multiply value and gate element-wise
        
        return output

---

**Question 9:** Show your completed code for the `GatedLayer` class after replacing all `None` values.

---

In [ ]:
# Create a GatedLayer for testing
# Input: 2 features (x1, x2), Output: 32 features
gated_layer = GatedLayer(input_dim=2, output_dim=32)

In [ ]:
# Run test function for the GatedLayer
test_gated_layer(gated_layer)

### Visualizing the Gate Behavior

Let's visualize how the gate values behave for different inputs.

In [ ]:
# Test with a batch of random inputs
test_inputs = torch.randn(5, 2)  # 5 samples, 2 features each
print("Test inputs shape:", test_inputs.shape)

# Get output
with torch.no_grad():
    output = gated_layer(test_inputs)
print("Output shape:", output.shape)

# Peek at gate values for the first sample
with torch.no_grad():
    gate_values = gated_layer.sigmoid(gated_layer.linear_gate(test_inputs[0:1]))
print("\nGate values for first sample (should be between 0 and 1):")
print(gate_values)

---

**Question 10:** Compare the `GatedLayer` to a standard approach of `nn.Linear` followed by `nn.ReLU`:

- How many learnable parameters does `GatedLayer(2, 32)` have? How does this compare to `nn.Linear(2, 32)` followed by `nn.ReLU()`?
- What can `GatedLayer` represent that `Linear + ReLU` cannot? (Hint: think about what happens when the gate is partially open)
- Why might gating be useful for learning complex decision boundaries?

---

## What's Next?

Our task continues in **Notebook HW1-C**, where we will build a complete neural network using the `GatedLayer` and train it on our wave dataset.